# Capstone — Procurement assistant with a router, four ML models, and a sourcing agent

A RAG chatbot where retrieval is only one of three lanes. Each turn is parsed
into a structured request, checked against the laptop catalogue, and routed to
whichever subsystem can actually answer it.

| Gate | Condition | Lane |
|---|---|---|
| **G0** | spec not in the catalogue, or a required field missing | correct &amp; ask — *no model call* |
| **G1** | definitional / historical question | RAG over Qdrant |
| **G2** | one fully-resolved candidate | ML scoring (4 models) |
| **G3** | a goal plus a constraint, no candidate | **sourcing agent** — plan / search / score / replan |
| **G4** | many requests at once | batch lane |

Gates are evaluated in order and the first match wins, so the **ordering is the
design**. G0 leads because a request naming hardware that does not exist must
never reach a model that would score it anyway — `"MacBook with Core i7"` encodes
without error (both values are individually valid) and returns a confident number
for a machine nobody sells. G2 precedes G3 because someone who names a machine
wants it *assessed*, not replaced.

Only G3 is an agent. Everything else is a single pass, which is deliberate: on
2×T4 an agent turn costs 5–8× the tokens of a plain answer.

---

**Session options — set before running anything:**

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU T4 x2** | 14B-AWQ needs both cards. P100 is compute capability 6.0 and has no AWQ kernels. |
| Internet | **On** | Weight downloads and the Qdrant round trip. |

Run the cells in order. The order matters twice: the embedding model is loaded
and then *freed* before vLLM starts (vLLM claims a fixed slice of VRAM and never
gives it back), and the ML models must exist on disk before the app boots.

## 1. Install

In [ ]:
# vLLM pins its own torch build, so install it first and let the rest resolve
# around it. Several minutes on a cold session.
!pip install -q vllm openai

!pip install -q llama-index-core llama-index-vector-stores-qdrant llama-index-embeddings-huggingface qdrant-client python-dotenv streamlit pyngrok xgboost scikit-learn joblib

## 2. Configuration

Every credential and model name lives **here and only here**; everything
downstream reads `os.environ`, and the vLLM and Streamlit subprocesses inherit
it. An earlier version of this notebook declared config separately in the
embedding cell and the app cell, which is how those two ended up pointing at two
different Qdrant clusters.

Set the secrets under **Add-ons → Secrets**. Nothing is hardcoded, so this
notebook is safe to commit.

In [ ]:
import os
from pathlib import Path


def secret(name, default=None, required=True):
    """Kaggle Secrets first, then the environment.

    Kaggle Secrets is the only one that exists on a fresh session; the
    environment fallback is what makes the same notebook runnable locally.
    """
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        pass

    value = os.environ.get(name, default)
    if required and not value:
        raise RuntimeError(f"{name} is not set. Add it under Add-ons -> Secrets.")
    return value


WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()

os.environ["QDRANT_URL"] = secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secret("QDRANT_API_KEY")
os.environ["QDRANT_COLLECTION"] = "capstone"
os.environ["NGROK_AUTHTOKEN"] = secret("NGROK_AUTHTOKEN")

# 1024-dim and multilingual, which the Indonesian source documents need.
# Changing this invalidates every stored vector: a query embedded with a
# different model still has the right dimensionality, so Qdrant accepts it and
# silently returns noise.
os.environ["EMBED_MODEL"] = "BAAI/bge-m3"
os.environ["EMBED_DEVICE"] = "cpu"

# The AWQ 4-bit build is ~10 GB and fits across two T4s with room for the KV
# cache; the fp16 14B is ~28 GB and would not.
os.environ["LLM_BACKEND"] = "local"
os.environ["HF_MODEL"] = "Qwen/Qwen2.5-14B-Instruct-AWQ"
os.environ["LOCAL_LLM_URL"] = "http://localhost:8000/v1"

os.environ["MODEL_DIR"] = str(WORK / "models")
# Kaggle's root filesystem is small and the default HF cache lives there, so a
# 10 GB download dies partway through.
os.environ["HF_HOME"] = str(WORK / "hf")

os.chdir(WORK)
print("Working dir:", WORK)
print("Collection :", os.environ["QDRANT_COLLECTION"])
print("Generation :", os.environ["HF_MODEL"], "(local vLLM)")

## 3. Write the project modules

Four files. They are written out rather than pasted inline so the notebook, the
Streamlit app, and the tests all import *the same* code — the alternative is two
copies of the routing rules that quietly disagree.

| File | Contents |
|---|---|
| `syntetic_data_complete.py` | the synthetic procurement dataset |
| `ml_models.py` | feature engineering, training, and `score_request` |
| `procurement_agent.py` | catalogue, extraction, router, sourcing agent |
| `app.py` | the Streamlit UI |

### 3a. Synthetic data

In [ ]:
%%writefile syntetic_data_complete.py
"""Synthetic procurement dataset for the four ML use cases in ml_applications.md.

Design notes
------------
The labels are *not* hard if/else rules. Each one is built as a latent score
from the features, pushed through a sigmoid (classification) or used as a
distribution mean (regression), and then sampled. That matters: a rule like
`if ram < 16: match = 0` makes the label a lookup table, so a tree model just
rediscovers the threshold and reports near-perfect accuracy that means nothing.
Sampling from a latent score leaves genuine Bayes error, so the metrics in
ml_applications.ipynb reflect a real decision surface.

The feature set also covers what ml_applications.md actually asks for --
travel frequency (UC2), per-model IT ticket history (UC3), and time since last
upgrade (UC4) -- which the first version of this script promised but never
generated.

FEATURE_COLUMNS and TARGET_COLUMNS are exported so the training notebook can
build X without ever touching another use case's label. All four targets live
in one table, so cross-target leakage is the easiest mistake to make here.
"""

import numpy as np
import pandas as pd

RANDOM_SEED = 42

# --------------------------------------------------------------------------
# Catalogue
# --------------------------------------------------------------------------

# compute_tier   1 = office/browser, 2 = general, 3 = heavy, 4 = workstation
# build_quality  0..1, drives how well the chassis survives being carried around
# portability    0..1, higher = lighter/more travel-friendly
LAPTOPS = [
    # brand,   model,            cpu,        gpu,          base_ram, base_ssd, base_price, tier, build, port
    ("Asus",   "ROG Zephyrus",   "Core i9",  "RTX 4070",   32,  1024, 35_000_000, 4, 0.55, 0.30),
    ("Asus",   "ExpertBook B9",  "Core i7",  "Iris Xe",    16,  1024, 25_000_000, 2, 0.80, 0.95),
    ("Asus",   "Vivobook 14",    "Core i5",  "Integrated",  8,   512, 10_500_000, 1, 0.50, 0.70),
    ("Lenovo", "ThinkPad P16",   "Core i9",  "RTX A2000",  32,  1024, 38_000_000, 4, 0.90, 0.25),
    ("Lenovo", "ThinkPad T14",   "Core i5",  "Integrated", 16,   512, 18_000_000, 2, 0.92, 0.75),
    ("Lenovo", "ThinkPad X1",    "Core i7",  "Iris Xe",    16,  1024, 28_000_000, 3, 0.90, 0.95),
    ("Lenovo", "IdeaPad Slim 3", "Core i3",  "Integrated",  8,   256,  8_000_000, 1, 0.45, 0.65),
    ("HP",     "ZBook Firefly",  "Core i7",  "RTX A500",   32,  1024, 30_000_000, 3, 0.80, 0.60),
    ("HP",     "OmniBook 5",     "Ryzen 7",  "Radeon",     16,   512, 17_500_000, 2, 0.65, 0.80),
    ("HP",     "ProBook 445",    "Ryzen 5",  "Radeon",      8,   512, 12_000_000, 1, 0.60, 0.70),
    ("Dell",   "Precision 5690", "Core i9",  "RTX 3500",   32,  2048, 42_000_000, 4, 0.85, 0.35),
    ("Dell",   "Latitude 7450",  "Core i7",  "Integrated", 16,   512, 22_000_000, 2, 0.85, 0.85),
    ("Dell",   "Inspiron 15",    "Core i5",  "Integrated",  8,   512,  9_500_000, 1, 0.50, 0.60),
    ("Apple",  "MacBook Pro 14", "M3 Pro",   "Apple GPU",  18,   512, 32_000_000, 4, 0.95, 0.85),
    ("Apple",  "MacBook Air 13", "M3",       "Apple GPU",   8,   256, 18_500_000, 2, 0.95, 1.00),
]

LAPTOP_COLUMNS = [
    "laptop_brand", "laptop_model", "cpu", "gpu",
    "base_ram_gb", "base_storage_gb", "base_price", "compute_tier",
    "build_quality", "portability",
]

# compute_need   what the role actually needs, on the same 1..4 scale as compute_tier
# travel         baseline trips per month, drives wear in UC2
# windows_locked share of the department that cannot use macOS (Excel macros, core banking apps)
DEPARTMENTS = {
    "IT":            {"compute_need": 3.4, "travel": 0.8, "windows_locked": 0.35, "budget_cap": 32_000_000},
    "Design":        {"compute_need": 3.6, "travel": 0.6, "windows_locked": 0.10, "budget_cap": 30_000_000},
    "Finance":       {"compute_need": 2.0, "travel": 0.7, "windows_locked": 0.90, "budget_cap": 20_000_000},
    "HR":            {"compute_need": 1.5, "travel": 0.5, "windows_locked": 0.60, "budget_cap": 15_000_000},
    "Sales":         {"compute_need": 1.8, "travel": 4.5, "windows_locked": 0.55, "budget_cap": 18_000_000},
    "Medical Staff": {"compute_need": 2.2, "travel": 1.2, "windows_locked": 0.70, "budget_cap": 22_000_000},
}

# prestige lifts what a role is expected to receive; multiplier scales the policy cap
SENIORITY = {
    "Staff":    {"prestige": 0.0, "cap_multiplier": 0.75},
    "Senior":   {"prestige": 0.6, "cap_multiplier": 1.00},
    "Manager":  {"prestige": 1.3, "cap_multiplier": 1.35},
    "Director": {"prestige": 2.0, "cap_multiplier": 1.90},
}

RAM_UPGRADES = [0, 8, 16]        # GB added on top of the base configuration
STORAGE_UPGRADES = [0, 512, 1024]
WARRANTY_YEARS = [1, 2, 3]

# Rough street cost of an upgrade, per unit, in IDR.
RAM_COST_PER_GB = 250_000
STORAGE_COST_PER_GB = 4_000


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_procurement_ml_data(num_records=4000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    catalogue = pd.DataFrame(LAPTOPS, columns=LAPTOP_COLUMNS)

    # Per-model reliability that the buyer cannot see directly. UC3's ticket
    # history is a noisy observation of it, which is what makes UC3 learnable
    # without simply handing over the answer.
    model_defect_rate = {
        row.laptop_model: float(np.clip(rng.normal(0.30 - 0.22 * row.build_quality, 0.05), 0.03, 0.60))
        for row in catalogue.itertuples()
    }

    picks = rng.integers(0, len(catalogue), size=num_records)
    rows = []

    for i in range(num_records):
        spec = catalogue.iloc[picks[i]]

        dept = rng.choice(list(DEPARTMENTS))
        dept_cfg = DEPARTMENTS[dept]
        role = rng.choice(list(SENIORITY), p=[0.45, 0.30, 0.18, 0.07])
        role_cfg = SENIORITY[role]

        # ---------------- configuration actually ordered ----------------
        ram = int(spec.base_ram_gb + rng.choice(RAM_UPGRADES, p=[0.60, 0.28, 0.12]))
        storage = int(spec.base_storage_gb + rng.choice(STORAGE_UPGRADES, p=[0.65, 0.25, 0.10]))
        warranty_years = int(rng.choice(WARRANTY_YEARS, p=[0.45, 0.35, 0.20]))

        config_price = (
            spec.base_price
            + (ram - spec.base_ram_gb) * RAM_COST_PER_GB
            + (storage - spec.base_storage_gb) * STORAGE_COST_PER_GB
            + (warranty_years - 1) * 1_200_000
        )

        # Vendors quote around the configured price; the spread is the thing
        # UC4 has to react to, so it is wider than a token +/-10%.
        historical_avg_price = int(config_price * rng.normal(0.97, 0.03))
        requested_unit_price = int(config_price * rng.normal(1.02, 0.09))
        quantity = int(rng.integers(1, 26))

        # ---------------- employee / request context ----------------
        tenure_months = int(np.clip(rng.gamma(2.2, 22), 1, 240))
        months_since_last_upgrade = int(np.clip(rng.normal(34, 14), 1, 96))
        travel_days_per_month = float(np.clip(rng.gamma(2.0, dept_cfg["travel"] / 2.0), 0, 22))
        prior_replacements = int(rng.poisson(0.45))

        vendor_risk_score = float(np.clip(rng.beta(2.2, 4.5), 0.01, 0.99))
        vendor_is_official = int(rng.random() < (0.75 - 0.5 * vendor_risk_score))

        policy_cap = dept_cfg["budget_cap"] * role_cfg["cap_multiplier"]
        # Centred well above the request so most departments can actually afford
        # what they ask for; a tighter sigma keeps wildly-over-budget requests
        # rare rather than routine.
        # The ceiling is high enough not to truncate the lognormal: at a 900M cap
        # roughly a fifth of rows piled up on the boundary as an identical value,
        # which is an artefact rather than a budget.
        dept_budget_remaining = int(np.clip(
            rng.lognormal(np.log(policy_cap * quantity * 1.8), 0.60),
            5_000_000, 6_000_000_000,
        ))
        is_urgent = int(rng.random() < 0.18)
        requires_windows = int(rng.random() < dept_cfg["windows_locked"])

        total_amount = requested_unit_price * quantity

        # ==============================================================
        # UC1 -- suitability. Under-spec hurts a lot, over-spec hurts a
        # little (wasted budget), which is the signal the assistant needs
        # to say "32 GB is overkill for this role".
        # ==============================================================
        capability = spec.compute_tier + 0.55 * np.log2(max(ram, 1) / 8.0)
        gap = capability - dept_cfg["compute_need"]
        fit = -1.55 * max(0.0, -gap) ** 1.5 - 0.42 * max(0.0, gap) ** 1.3

        prestige_gap = role_cfg["prestige"] - (requested_unit_price / 12_000_000.0)

        # Intercept calibrated so roughly 70% of past deployments succeeded --
        # a 30% success rate would not describe any real procurement function.
        uc1_score = (
            3.75
            + 1.5 * fit
            - 0.55 * abs(prestige_gap)
            - 2.1 * (requires_windows and spec.laptop_brand == "Apple")
            + 1.25 * spec.portability * (travel_days_per_month / 10.0)
            - 0.9 * (1.0 - spec.build_quality) * (travel_days_per_month / 10.0)
            + 0.30 * (storage >= 512)
        )
        is_successful_match = int(rng.random() < _sigmoid(uc1_score))

        # ==============================================================
        # UC2 -- months until first major failure. Gamma keeps it positive
        # and right-skewed, which is how hardware survival actually looks.
        # ==============================================================
        # Centred so the fleet mean lands near a ~44-month refresh cycle.
        expected_life = (
            18.0
            + 32.0 * spec.build_quality
            - 1.35 * travel_days_per_month
            - 3.2 * prior_replacements
            + 5.0 * warranty_years
            - 14.0 * model_defect_rate[spec.laptop_model]
        )
        expected_life = float(np.clip(expected_life, 8.0, 84.0))
        # Gamma sd is mean/sqrt(shape). At shape=9 that is ~15 months against a
        # systematic spread of about the same size, leaving R^2 near 0.2 -- the
        # noise drowns the signal. shape=25 puts sd near 9 months, so a model
        # can recover the durability signal while real error remains.
        shape = 25.0
        months_until_failure = float(np.clip(
            rng.gamma(shape, expected_life / shape), 3.0, 120.0,
        ))

        # Observable proxy for reliability: tickets logged against this model
        # across the fleet. Correlated with the hidden defect rate, not equal.
        it_tickets_last_year = int(rng.poisson(
            np.clip(6.0 * model_defect_rate[spec.laptop_model] + 0.12 * travel_days_per_month, 0.1, 12.0)
        ))

        # ==============================================================
        # UC3 -- 3-year total cost of ownership.
        # ==============================================================
        setup_cost = rng.normal(1_500_000, 250_000)
        ecosystem_tax = 3_000_000 if spec.laptop_brand == "Apple" else 0
        # Repairs inside the 36-month window, minus whatever warranty absorbs.
        expected_repairs = max(0.0, (36.0 - months_until_failure) / 36.0) * 2.0
        covered = min(warranty_years, 3) / 3.0
        repair_cost = expected_repairs * 4_200_000 * (1.0 - 0.65 * covered)
        # Lost productivity while a machine is in for service. Scales with the
        # seniority of whoever is idle, which is why it is not a flat rate.
        downtime_rate = rng.normal(420_000, 90_000) * (1.0 + 0.45 * role_cfg["prestige"])
        downtime_cost = it_tickets_last_year * 3 * downtime_rate

        # Everything the purchase price does NOT already tell you.
        opex_3_years_idr = float(setup_cost + ecosystem_tax + repair_cost + downtime_cost)
        tco_3_years_idr = float(requested_unit_price + opex_3_years_idr)

        # ==============================================================
        # UC4 -- approval. Deliberately imbalanced (~20-25% rejected) so the
        # notebook's imbalance handling is doing real work.
        # ==============================================================
        budget_ratio = total_amount / max(dept_budget_remaining, 1)
        price_premium = (requested_unit_price - historical_avg_price) / max(historical_avg_price, 1)
        cap_overrun = (requested_unit_price - policy_cap) / policy_cap

        # Calibrated to ~78% approved. The imbalance is the point: a balanced
        # approval label would make the notebook's class-weighting a no-op.
        uc4_score = (
            4.05
            # Clipped: budget_ratio has a long right tail, and an unclipped
            # hinge sent those rows to a logit near -90, making them
            # deterministic rejections with no Bayes error left to learn.
            - 2.8 * min(max(0.0, budget_ratio - 0.55), 2.5)
            - 3.4 * min(max(0.0, cap_overrun), 1.5)
            - 4.5 * max(0.0, price_premium - 0.05)
            # Linear, not hinged: vendor_risk_score is Beta(2.2, 4.5), so a
            # hinge at 0.55 sat above the 75th percentile and the term was
            # almost always zero -- approval ended up barely reacting to risk.
            - 3.2 * vendor_risk_score
            + 0.55 * vendor_is_official
            + 0.75 * is_urgent
            + 0.014 * months_since_last_upgrade
            + 0.30 * role_cfg["prestige"]
        )
        is_approved = int(rng.random() < _sigmoid(uc4_score))

        rows.append({
            "request_id": f"REQ-2026-{i:05d}",
            # --- request context
            "department": dept,
            "seniority_level": role,
            "employee_tenure_months": tenure_months,
            "months_since_last_upgrade": months_since_last_upgrade,
            "travel_days_per_month": round(travel_days_per_month, 2),
            "prior_replacements": prior_replacements,
            "requires_windows": requires_windows,
            "is_urgent": is_urgent,
            # --- product
            "laptop_brand": spec.laptop_brand,
            "laptop_model": spec.laptop_model,
            "cpu": spec.cpu,
            "gpu": spec.gpu,
            "ram_gb": ram,
            "storage_gb": storage,
            "compute_tier": int(spec.compute_tier),
            "build_quality": round(float(spec.build_quality), 2),
            "portability": round(float(spec.portability), 2),
            "warranty_years": warranty_years,
            "it_tickets_last_year": it_tickets_last_year,
            # --- commercial
            "quantity": quantity,
            "requested_unit_price": requested_unit_price,
            "historical_avg_price": historical_avg_price,
            "total_amount": total_amount,
            "vendor_risk_score": round(vendor_risk_score, 3),
            "vendor_is_official": vendor_is_official,
            "dept_budget_remaining": dept_budget_remaining,
            "dept_policy_cap": int(policy_cap),
            # --- targets
            "target_uc1_is_match": is_successful_match,
            "target_uc2_months_to_failure": round(months_until_failure, 1),
            "target_uc3_tco_idr": round(tco_3_years_idr, 0),
            # The modelling target for UC3. Total TCO is dominated by
            # requested_unit_price, which is a *feature* -- predicting it scores
            # R^2 ~ 0.997 by echoing an input back and tells finance nothing they
            # did not already know. The operating cost is the unknown half.
            "target_uc3_opex_idr": round(opex_3_years_idr, 0),
            "target_uc4_is_approved": is_approved,
        })

    return pd.DataFrame(rows)


# Exported so the notebook can assemble X without ever reaching into another
# use case's label. All four targets share one table, which makes cross-target
# leakage the easiest mistake available here.
TARGET_COLUMNS = [
    "target_uc1_is_match",
    "target_uc2_months_to_failure",
    "target_uc3_tco_idr",
    "target_uc3_opex_idr",
    "target_uc4_is_approved",
]

ID_COLUMNS = ["request_id"]

CATEGORICAL_FEATURES = [
    "department", "seniority_level", "laptop_brand", "laptop_model", "cpu", "gpu",
]

NUMERIC_FEATURES = [
    "employee_tenure_months", "months_since_last_upgrade", "travel_days_per_month",
    "prior_replacements", "requires_windows", "is_urgent",
    "ram_gb", "storage_gb", "compute_tier", "build_quality", "portability",
    "warranty_years", "it_tickets_last_year",
    "quantity", "requested_unit_price", "historical_avg_price", "total_amount",
    "vendor_risk_score", "vendor_is_official", "dept_budget_remaining", "dept_policy_cap",
]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES


if __name__ == "__main__":
    df = generate_procurement_ml_data(4000)
    df.to_csv("laptop_procurement_ml_training_data.csv", index=False)

    print("Dataset shape:", df.shape)
    print("\nLabel balance / distribution:")
    print(f"  UC1 match rate      : {df.target_uc1_is_match.mean():.1%}")
    print(f"  UC2 months to fail  : mean {df.target_uc2_months_to_failure.mean():.1f}, "
          f"sd {df.target_uc2_months_to_failure.std():.1f}")
    print(f"  UC3 TCO (IDR)       : mean {df.target_uc3_tco_idr.mean():,.0f}")
    print(f"  UC3 opex (IDR)      : mean {df.target_uc3_opex_idr.mean():,.0f}, "
          f"sd {df.target_uc3_opex_idr.std():,.0f}")
    print(f"  UC4 approval rate   : {df.target_uc4_is_approved.mean():.1%}")
    print(f"\n{len(FEATURE_COLUMNS)} features, {len(TARGET_COLUMNS)} targets")

### 3b. ML models

The labels here are sampled from latent scores rather than written as if/else
rules. That matters: a rule like `if ram < 16: match = 0` makes the label a
lookup table, and a tree recovers it at ~100% accuracy — a number that means
nothing. Sampling leaves real Bayes error, so the metrics below describe a
decision surface.

In [ ]:
%%writefile ml_models.py
"""Train, persist and serve the four procurement models.

ml_applications.ipynb explores these -- feature engineering, the leakage guard,
hyperparameter search, and the metric comparison that picked each winner. This
module is the production side of the same work: it fits the chosen models with
the hyperparameters that search settled on, saves them, and exposes one
`score_request` entry point for the agent to call.

Hyperparameters are fixed rather than re-searched. Re-running RandomizedSearchCV
inside the serving path would add minutes to every cold start to rediscover
values already known.
"""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

from syntetic_data_complete import (
    CATEGORICAL_FEATURES,
    DEPARTMENTS,
    ID_COLUMNS,
    NUMERIC_FEATURES,
    TARGET_COLUMNS,
    generate_procurement_ml_data,
)

RANDOM_STATE = 42
MODEL_DIR = Path("models")

DEPT_COMPUTE_NEED = {name: cfg["compute_need"] for name, cfg in DEPARTMENTS.items()}

ENGINEERED = [
    "price_premium_ratio", "budget_utilisation", "cap_overrun_ratio",
    "spec_gap", "is_overspec", "is_underspec",
    "price_per_ram_gb", "price_per_storage_gb",
    "tickets_per_travel_day", "wear_exposure",
]

NUMERIC_ALL = NUMERIC_FEATURES + ENGINEERED


def engineer_features(frame):
    """Derive ratio and spec-adequacy features. Reads inputs only, never a label."""
    out = frame.copy()

    out["price_premium_ratio"] = (
        (out.requested_unit_price - out.historical_avg_price) / out.historical_avg_price
    )
    out["budget_utilisation"] = out.total_amount / out.dept_budget_remaining
    out["cap_overrun_ratio"] = (
        (out.requested_unit_price - out.dept_policy_cap) / out.dept_policy_cap
    )

    # The signed distance between what the machine can do and what the role
    # needs. Positive is over-provisioned, negative is under -- this is the
    # number the assistant turns into "32 GB berlebihan".
    capability = out.compute_tier + 0.55 * np.log2(out.ram_gb.clip(lower=1) / 8.0)
    out["spec_gap"] = capability - out.department.map(DEPT_COMPUTE_NEED)
    out["is_overspec"] = (out.spec_gap > 0.75).astype(int)
    out["is_underspec"] = (out.spec_gap < -0.75).astype(int)

    out["price_per_ram_gb"] = out.requested_unit_price / out.ram_gb
    out["price_per_storage_gb"] = out.requested_unit_price / out.storage_gb

    out["tickets_per_travel_day"] = out.it_tickets_last_year / (out.travel_days_per_month + 1.0)
    out["wear_exposure"] = out.travel_days_per_month * (1.0 - out.build_quality)

    return out


def build_matrix(frame):
    """Assemble X, dropping every target so no model can see another's label."""
    banned = set(TARGET_COLUMNS) | set(ID_COLUMNS)
    cols = [c for c in CATEGORICAL_FEATURES + NUMERIC_ALL if c not in banned]

    leaked = banned.intersection(cols)
    if leaked:
        raise AssertionError(f"target/id column leaked into features: {leaked}")

    return frame[cols]


def _preprocessor(scale_numeric):
    """scale_numeric=True for linear models; a no-op for trees, which split on rank."""
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", StandardScaler() if scale_numeric else "passthrough", NUMERIC_ALL),
    ])


def _pipe(model, scale_numeric=False):
    return Pipeline([("prep", _preprocessor(scale_numeric)), ("model", model)])


def train_all(n_records=6000, out_dir=MODEL_DIR, verbose=True):
    """Fit all four models plus the serving defaults, and persist the bundle."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = engineer_features(generate_procurement_ml_data(n_records))
    X = build_matrix(df)
    report = {}

    # --- UC1 suitability: RandomForest won on ROC-AUC / PR-AUC.
    y = df.target_uc1_is_match
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    uc1 = _pipe(RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_leaf=4, max_features="log2",
        random_state=RANDOM_STATE, n_jobs=-1,
    )).fit(Xtr, ytr)
    report["uc1_pr_auc"] = average_precision_score(yte, uc1.predict_proba(Xte)[:, 1])

    # --- UC4 approval: XGBoost with the loss reweighted for the 30% minority.
    y = df.target_uc4_is_approved
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    uc4 = _pipe(XGBClassifier(
        n_estimators=350, max_depth=4, learning_rate=0.06, subsample=0.85,
        colsample_bytree=0.8, min_child_weight=3, reg_lambda=3.0,
        scale_pos_weight=spw, eval_metric="logloss", tree_method="hist",
        random_state=RANDOM_STATE,
    )).fit(Xtr, ytr)
    report["uc4_pr_auc"] = average_precision_score(yte, uc4.predict_proba(Xte)[:, 1])

    # Threshold picked on TRAIN only -- choosing it on the test split would be
    # a second, quieter form of leakage.
    tr_proba = uc4.predict_proba(Xtr)[:, 1]
    prec, rec, thr = precision_recall_curve(ytr, tr_proba)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)
    uc4_threshold = float(thr[np.argmax(f1[:-1])])
    report["uc4_threshold"] = uc4_threshold
    report["uc4_f1"] = f1_score(yte, (uc4.predict_proba(Xte)[:, 1] >= uc4_threshold).astype(int))

    # --- UC2 lifespan: ridge beat both ensembles; the generator's survival term
    # is linear in the features, so a linear model is correctly specified.
    y = df.target_uc2_months_to_failure
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    uc2 = _pipe(Ridge(alpha=10.0, random_state=RANDOM_STATE), scale_numeric=True).fit(Xtr, ytr)
    report["uc2_r2"] = r2_score(yte, uc2.predict(Xte))

    # --- UC3 operating cost, NOT total TCO. Total TCO is dominated by
    # requested_unit_price, which is an input -- predicting it scores R^2 ~ 0.99
    # by echoing a feature back.
    y = df.target_uc3_opex_idr
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    uc3 = _pipe(XGBRegressor(
        n_estimators=550, max_depth=6, learning_rate=0.06, subsample=0.85,
        colsample_bytree=0.8, min_child_weight=3, reg_lambda=1.0,
        tree_method="hist", random_state=RANDOM_STATE,
    )).fit(Xtr, ytr)
    report["uc3_r2"] = r2_score(yte, uc3.predict(Xte))

    # Median/mode of every input, so a partially-specified request can still be
    # scored -- with the caller told exactly which fields were invented.
    defaults = {
        c: (df[c].mode()[0] if c in CATEGORICAL_FEATURES else float(df[c].median()))
        for c in CATEGORICAL_FEATURES + NUMERIC_FEATURES
    }

    bundle = {
        "uc1": uc1, "uc2": uc2, "uc3": uc3, "uc4": uc4,
        "uc4_threshold": uc4_threshold,
        "defaults": defaults,
        "report": report,
    }
    joblib.dump(bundle, out_dir / "procurement_models.joblib")

    if verbose:
        print(f"Saved bundle to {out_dir / 'procurement_models.joblib'}")
        for k, v in report.items():
            print(f"  {k:16} {v:.4f}")

    return bundle


def load_all(out_dir=MODEL_DIR):
    path = Path(out_dir) / "procurement_models.joblib"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found -- run ml_models.train_all() first.")
    return joblib.load(path)


# Fields a caller must supply for the scores to mean anything. Everything else
# falls back to a median, which is fine for wear_exposure and misleading for a
# budget -- hence the split.
REQUIRED_FOR_SUITABILITY = ["department", "ram_gb", "compute_tier"]
REQUIRED_FOR_APPROVAL = ["requested_unit_price", "dept_budget_remaining", "dept_policy_cap"]

# A few fields have an obviously safe fallback that the training median is not.
# `quantity` is the one that matters: the median order is ~13 units, so an
# unspecified quantity silently turned a 30M laptop into a 390M request and
# drove every approval probability to near zero. One unit is what a person
# asking about "a laptop" means.
SAFE_DEFAULTS = {"quantity": 1}


def score_request(bundle, fields):
    """Score a partially-specified request; report what had to be assumed.

    `approval_*` is withheld when the budget context is missing rather than
    computed from a median budget, which would look authoritative and mean
    nothing.
    """
    defaults = bundle["defaults"]

    unknown = set(fields) - set(defaults)
    if unknown:
        raise KeyError(f"unknown field(s): {sorted(unknown)}")

    row = {**defaults, **SAFE_DEFAULTS, **{k: v for k, v in fields.items() if v is not None}}
    supplied = {k for k, v in fields.items() if v is not None}
    assumed = sorted(set(defaults) - supplied)

    # A quoted price with no historical reference is not a 36% premium over the
    # fleet median -- it is simply the price. Anchoring the two together keeps
    # price_premium_ratio at zero instead of inventing an overcharge.
    if "historical_avg_price" not in supplied and "requested_unit_price" in supplied:
        row["historical_avg_price"] = row["requested_unit_price"]

    row["total_amount"] = row["requested_unit_price"] * row["quantity"]

    frame = engineer_features(pd.DataFrame([row]))
    matrix = build_matrix(frame)

    result = {
        "suitability_proba": float(bundle["uc1"].predict_proba(matrix)[0, 1]),
        "spec_gap": float(frame.spec_gap.iloc[0]),
        "expected_lifespan_months": float(bundle["uc2"].predict(matrix)[0]),
        "expected_opex_idr": float(bundle["uc3"].predict(matrix)[0]),
        "assumed_fields": assumed,
    }
    result["expected_total_tco_idr"] = result["expected_opex_idr"] + row["total_amount"]

    if all(f in supplied for f in REQUIRED_FOR_APPROVAL):
        proba = float(bundle["uc4"].predict_proba(matrix)[0, 1])
        result["approval_proba"] = proba
        result["approval_decision"] = (
            "likely approved" if proba >= bundle["uc4_threshold"] else "likely rejected"
        )
    else:
        result["approval_proba"] = None
        result["approval_decision"] = "unknown"
        result["approval_blocked_on"] = [f for f in REQUIRED_FOR_APPROVAL if f not in supplied]

    return result


if __name__ == "__main__":
    train_all()

### 3c. Router and sourcing agent

`procurement_agent.py` holds the parts the artifact sketch described: catalogue
validation for G0, the ordered router, and the plan/search/score/replan loop with
its fixed relaxation ladder and the hard floor it never crosses.

In [ ]:
%%writefile procurement_agent.py
"""Router and sourcing agent for the procurement assistant.

Three lanes sit behind one chat box, and the router decides which one a turn
belongs to. Most turns are not agentic and should not pay for an agent; one kind
genuinely is.

    G0  spec not in the catalogue, or a required field missing  -> correct & ask
    G1  definitional / historical question about the documents  -> plain RAG
    G2  one fully-resolved candidate                            -> ML scoring
    G3  a goal plus a constraint, but no candidate              -> sourcing agent
    G4  scope is a set of requests rather than one              -> batch lane
    --  nothing matched                                         -> clarify

Gates are evaluated in order and the first match wins, so the ordering is the
design. G0 sits ahead of everything because a request naming hardware that does
not exist must never reach a model that would score it anyway: "MacBook with
Core i7" encodes cleanly (both values are individually valid) and yields a
confident number for a machine nobody sells.

G2 sits ahead of G3 deliberately. Someone who names a specific machine wants it
assessed, not replaced; searching for alternatives is a different act from
scoring what was asked about.
"""

import json
import re
from dataclasses import dataclass, field, asdict

from ml_models import DEPT_COMPUTE_NEED, score_request
from syntetic_data_complete import DEPARTMENTS, LAPTOPS, SENIORITY

# --------------------------------------------------------------------------
# Catalogue -- the ground truth G0 validates against
# --------------------------------------------------------------------------

CATALOGUE = [
    {
        "laptop_brand": b, "laptop_model": m, "cpu": c, "gpu": g,
        "base_ram_gb": ram, "base_storage_gb": ssd, "base_price": price,
        "compute_tier": tier, "build_quality": build, "portability": port,
    }
    for (b, m, c, g, ram, ssd, price, tier, build, port) in LAPTOPS
]

BRANDS = sorted({row["laptop_brand"] for row in CATALOGUE})
MODELS = sorted({row["laptop_model"] for row in CATALOGUE})
DEPARTMENT_NAMES = sorted(DEPARTMENTS)
SENIORITY_NAMES = list(SENIORITY)

RAM_OPTIONS = [8, 16, 24, 32, 48]

# Indonesian role abbreviations that appear in real requests. Without this the
# extractor has to guess what "AMGR" means, and a wrong seniority silently
# changes both the policy cap and the suitability score.
ROLE_ALIASES = {
    "staff": "Staff", "stf": "Staff", "officer": "Staff", "ofc": "Staff",
    "senior": "Senior", "snr": "Senior", "sr": "Senior",
    "asisten manager": "Manager", "asst manager": "Manager", "amgr": "Manager",
    "manager": "Manager", "mgr": "Manager", "manajer": "Manager",
    "director": "Director", "direktur": "Director", "dir": "Director",
}


def models_for_brand(brand):
    return sorted({r["laptop_model"] for r in CATALOGUE if r["laptop_brand"] == brand})


def cpus_for_brand(brand):
    return sorted({r["cpu"] for r in CATALOGUE if r["laptop_brand"] == brand})


def catalogue_row(brand=None, model=None):
    for row in CATALOGUE:
        if model and row["laptop_model"] == model:
            return row
        if brand and not model and row["laptop_brand"] == brand:
            return row
    return None


# --------------------------------------------------------------------------
# Request state -- slots accumulate across turns
# --------------------------------------------------------------------------

@dataclass
class RequestSlots:
    """What the assistant knows so far.

    Slots survive across turns: the laptop arrives in turn 1 and the budget in
    turn 5. Without one home for this object the router reads a half-built
    request and mis-gates it.
    """
    department: str = None
    seniority_level: str = None
    laptop_brand: str = None
    laptop_model: str = None
    cpu: str = None
    gpu: str = None
    ram_gb: int = None
    storage_gb: int = None
    requested_unit_price: float = None
    quantity: int = None
    dept_budget_remaining: float = None
    intent: str = None            # "assess" | "find" | "ask" | "batch"
    free_text: str = ""

    def merge(self, updates):
        for key, value in (updates or {}).items():
            if value is not None and hasattr(self, key):
                setattr(self, key, value)
        return self

    def has_candidate(self):
        """A candidate is resolved once we know which machine is meant."""
        return self.laptop_model is not None or (
            self.laptop_brand is not None and self.ram_gb is not None
        )

    def has_constraint(self):
        return self.requested_unit_price is not None or self.dept_budget_remaining is not None

    def as_scoring_fields(self):
        """Project the slots onto the feature names ml_models expects."""
        row = catalogue_row(self.laptop_brand, self.laptop_model)
        fields = {
            "department": self.department,
            "seniority_level": self.seniority_level,
            "laptop_brand": self.laptop_brand,
            "laptop_model": self.laptop_model,
            "cpu": self.cpu,
            "gpu": self.gpu,
            "ram_gb": self.ram_gb,
            "storage_gb": self.storage_gb,
            "requested_unit_price": self.requested_unit_price,
            "quantity": self.quantity,
            "dept_budget_remaining": self.dept_budget_remaining,
        }
        if row:
            fields.setdefault("compute_tier", row["compute_tier"])
            fields["compute_tier"] = row["compute_tier"]
            fields["build_quality"] = row["build_quality"]
            fields["portability"] = row["portability"]
        if self.department:
            cfg = DEPARTMENTS[self.department]
            mult = SENIORITY.get(self.seniority_level, SENIORITY["Staff"])["cap_multiplier"]
            fields["dept_policy_cap"] = cfg["budget_cap"] * mult
        return {k: v for k, v in fields.items() if v is not None}


# --------------------------------------------------------------------------
# Extraction (LLM, schema-constrained)
# --------------------------------------------------------------------------

EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "department":      {"type": ["string", "null"], "enum": DEPARTMENT_NAMES + [None]},
        "seniority_level": {"type": ["string", "null"], "enum": SENIORITY_NAMES + [None]},
        "laptop_brand":    {"type": ["string", "null"], "enum": BRANDS + [None]},
        "laptop_model":    {"type": ["string", "null"], "enum": MODELS + [None]},
        "cpu_text":        {"type": ["string", "null"]},
        "ram_gb":          {"type": ["integer", "null"]},
        "storage_gb":      {"type": ["integer", "null"]},
        "requested_unit_price": {"type": ["number", "null"]},
        "quantity":        {"type": ["integer", "null"]},
        "budget_idr":      {"type": ["number", "null"]},
        "intent": {"type": "string", "enum": ["assess", "find", "ask", "batch"]},
    },
    "required": ["intent"],
}

EXTRACTION_PROMPT = """Extract procurement request fields from the user message.
Reply with JSON only.

Rules:
- Use null for anything the message does not state. Never invent a value.
- `intent`: "assess" when a specific laptop is named for evaluation, "find" when
  the user wants options or a recommendation, "ask" for questions about
  documents or past purchases, "batch" when the subject is many requests at once.
- `budget_idr` is a spending limit ("budget 5 juta"); `requested_unit_price` is
  the price of a named machine. "5 juta" means 5000000.
- Copy the CPU verbatim into `cpu_text` (e.g. "intel i7", "M3 Pro").

Known departments: {departments}
Known seniority levels: {seniority} (AMGR / Asisten Manager = Manager, Officer = Staff)

Message: {message}"""


def normalise_role(text):
    """Map Indonesian role abbreviations onto the trained seniority levels."""
    if not text:
        return None
    lowered = text.lower()
    for alias, level in sorted(ROLE_ALIASES.items(), key=lambda kv: -len(kv[0])):
        if re.search(rf"\b{re.escape(alias)}\b", lowered):
            return level
    return None


def normalise_cpu(cpu_text, brand=None):
    """Resolve free-text CPU against the catalogue, per brand where known."""
    if not cpu_text:
        return None
    t = cpu_text.lower().replace("-", " ")
    pool = cpus_for_brand(brand) if brand else sorted({r["cpu"] for r in CATALOGUE})

    for cpu in pool:
        if cpu.lower() in t or t in cpu.lower():
            return cpu
    for cpu in pool:
        tail = cpu.lower().split()[-1]          # "i7" from "Core i7", "pro" from "M3 Pro"
        if re.search(rf"\b{re.escape(tail)}\b", t):
            return cpu
    return None


def extract_slots(client, model, message, current=None, temperature=0.0):
    """Ask the LLM for a structured request, constrained to the catalogue's enums.

    Guided decoding is what keeps the extractor inside the vocabulary the models
    were trained on. Without it the LLM invents a department, one-hot encoding
    silently drops it, and every downstream score drifts toward the mean with no
    error raised anywhere.
    """
    slots = current or RequestSlots()
    prompt = EXTRACTION_PROMPT.format(
        departments=", ".join(DEPARTMENT_NAMES),
        seniority=", ".join(SENIORITY_NAMES),
        message=message,
    )

    kwargs = dict(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=temperature,
    )
    try:
        response = client.chat.completions.create(
            **kwargs, extra_body={"guided_json": EXTRACTION_SCHEMA}
        )
    except Exception:
        # Older vLLM builds and non-vLLM servers reject guided_json; fall back to
        # unconstrained decoding and validate the JSON ourselves.
        response = client.chat.completions.create(**kwargs)

    raw = response.choices[0].message.content or "{}"
    match = re.search(r"\{.*\}", raw, re.S)
    payload = json.loads(match.group()) if match else {}

    brand = payload.get("laptop_brand")
    model_name = payload.get("laptop_model")
    if model_name and not brand:
        row = catalogue_row(model=model_name)
        brand = row["laptop_brand"] if row else None

    updates = {
        "department": payload.get("department"),
        "seniority_level": payload.get("seniority_level") or normalise_role(message),
        "laptop_brand": brand,
        "laptop_model": model_name,
        "ram_gb": payload.get("ram_gb"),
        "storage_gb": payload.get("storage_gb"),
        "requested_unit_price": payload.get("requested_unit_price"),
        "quantity": payload.get("quantity"),
        "dept_budget_remaining": payload.get("budget_idr"),
        "intent": payload.get("intent"),
    }
    slots.merge(updates)
    slots.free_text = message

    # Kept as raw text so validation can report the mismatch the user typed,
    # rather than a silently corrected value.
    slots._cpu_text = payload.get("cpu_text")
    slots.cpu = normalise_cpu(payload.get("cpu_text"), slots.laptop_brand)
    if slots.laptop_model:
        row = catalogue_row(model=slots.laptop_model)
        slots.gpu = row["gpu"] if row else None
    return slots


# --------------------------------------------------------------------------
# G0 -- catalogue validation
# --------------------------------------------------------------------------

@dataclass
class Validation:
    ok: bool
    corrections: list = field(default_factory=list)
    missing: list = field(default_factory=list)


def validate_slots(slots):
    """Reject configurations the catalogue does not contain.

    The failure this exists for is quiet: "Core i7" is a real category and
    "Apple" is a real brand, so an Apple/Core i7 request encodes without error
    and scores like any other row -- for a machine that has never been sold.
    """
    corrections, missing = [], []

    if slots.laptop_brand and slots.laptop_brand not in BRANDS:
        corrections.append(
            f"{slots.laptop_brand} is not in the catalogue. Available brands: "
            f"{', '.join(BRANDS)}."
        )

    if slots.laptop_model and slots.laptop_model not in MODELS:
        corrections.append(f"{slots.laptop_model} is not a catalogue model.")

    if slots.laptop_brand in BRANDS and slots.laptop_model:
        if slots.laptop_model not in models_for_brand(slots.laptop_brand):
            corrections.append(
                f"{slots.laptop_brand} does not ship the {slots.laptop_model}. "
                f"{slots.laptop_brand} models: {', '.join(models_for_brand(slots.laptop_brand))}."
            )

    raw_cpu = getattr(slots, "_cpu_text", None)
    if raw_cpu and slots.laptop_brand in BRANDS and slots.cpu is None:
        corrections.append(
            f"{slots.laptop_brand} does not ship a \"{raw_cpu}\" configuration. "
            f"{slots.laptop_brand} options: {', '.join(cpus_for_brand(slots.laptop_brand))}."
        )

    if slots.ram_gb is not None and slots.ram_gb not in RAM_OPTIONS:
        corrections.append(
            f"{slots.ram_gb} GB is not a configurable option. Available: "
            f"{', '.join(str(r) for r in RAM_OPTIONS)} GB."
        )

    if slots.department is None:
        missing.append("department")

    return Validation(ok=not corrections and not missing,
                      corrections=corrections, missing=missing)


# --------------------------------------------------------------------------
# Router
# --------------------------------------------------------------------------

GATE_DESCRIPTIONS = {
    "G0": "catalogue mismatch or missing required field -> correct and ask",
    "G1": "definitional / historical question -> RAG",
    "G2": "one resolved candidate -> ML scoring",
    "G3": "goal plus constraint, no candidate -> sourcing agent",
    "G4": "many requests at once -> batch lane",
    "CLARIFY": "nothing matched -> ask",
}


def route(slots):
    """Return the first gate that matches. Order is the design; see module docstring."""
    validation = validate_slots(slots)

    # G0 -- but a pure question does not need a department to be answerable.
    if validation.corrections:
        return "G0", validation
    if slots.intent in ("assess", "find") and validation.missing:
        return "G0", validation

    if slots.intent == "batch":
        return "G4", validation
    if slots.intent == "ask":
        return "G1", validation
    if slots.has_candidate() and slots.intent == "assess":
        return "G2", validation
    if slots.has_constraint() or slots.intent == "find":
        return "G3", validation
    return "CLARIFY", validation


# --------------------------------------------------------------------------
# Sourcing agent -- plan / search / score / evaluate / replan
# --------------------------------------------------------------------------

RELAXATION_LADDER = [
    ("price_band", "widened the price band by 15%"),
    ("drop_specs", "dropped non-binding specs (storage, screen)"),
    ("refurbished", "admitted refurbished and prior-generation units"),
    ("compute_tier", "lowered the compute tier (still at or above the role's floor)"),
]

MAX_RUNGS = len(RELAXATION_LADDER)


@dataclass
class Constraints:
    max_price: float = None
    min_ram: int = None
    allow_refurbished: bool = False
    price_multiplier: float = 1.0
    relax_specs: bool = False
    allow_lower_tier: bool = False

    def effective_price(self):
        return None if self.max_price is None else self.max_price * self.price_multiplier


class SourcingAgent:
    """Plan -> search -> score -> evaluate, relaxing one rung at a time.

    The loop is what distinguishes this from a chain: when nothing satisfies the
    constraint set, it re-enters planning with a weakened set rather than
    returning empty. The ladder is fixed and every descent is reported, so a
    procurement officer can see exactly which rule was bent and in what order.
    """

    def __init__(self, bundle, retriever=None):
        self.bundle = bundle
        self.retriever = retriever

    # -- the floor. None of these is ever relaxed. --------------------------
    def violates_floor(self, slots, candidate, scores):
        reasons = []

        cap = slots.as_scoring_fields().get("dept_policy_cap")
        if cap is not None and candidate["price"] > cap:
            reasons.append("exceeds the departmental policy cap")

        if scores["spec_gap"] < 0:
            reasons.append("under-provisioned for the role")

        if candidate.get("is_suspected_scam"):
            reasons.append("listing is flagged as a suspected scam")

        # A department locked to Windows tooling cannot be handed macOS; that is
        # a wrong answer, not a trade-off to weigh against price.
        if candidate["row"]["laptop_brand"] == "Apple":
            if DEPARTMENTS.get(slots.department, {}).get("windows_locked", 0) >= 0.85:
                reasons.append("department is locked to Windows tooling")

        return reasons

    def search(self, slots, constraints):
        """Enumerate configurations the catalogue can actually supply."""
        budget = constraints.effective_price()
        out = []

        for row in CATALOGUE:
            for ram in RAM_OPTIONS:
                if ram < row["base_ram_gb"]:
                    continue
                if constraints.min_ram and ram < constraints.min_ram and not constraints.relax_specs:
                    continue

                price = row["base_price"] + (ram - row["base_ram_gb"]) * 250_000
                if constraints.allow_refurbished:
                    price *= 0.78
                if budget is not None and price > budget:
                    continue

                out.append({
                    "row": row,
                    "ram_gb": ram,
                    "price": price,
                    "refurbished": constraints.allow_refurbished,
                    "is_suspected_scam": False,
                })
        return out

    def score(self, slots, candidate):
        fields = slots.as_scoring_fields()
        fields.update({
            "laptop_brand": candidate["row"]["laptop_brand"],
            "laptop_model": candidate["row"]["laptop_model"],
            "cpu": candidate["row"]["cpu"],
            "gpu": candidate["row"]["gpu"],
            "compute_tier": candidate["row"]["compute_tier"],
            "build_quality": candidate["row"]["build_quality"],
            "portability": candidate["row"]["portability"],
            "ram_gb": candidate["ram_gb"],
            "storage_gb": candidate["row"]["base_storage_gb"],
            "requested_unit_price": candidate["price"],
        })
        return score_request(self.bundle, fields)

    def run(self, slots, min_suitability=0.55, max_rungs=MAX_RUNGS):
        constraints = Constraints(
            max_price=slots.dept_budget_remaining or slots.requested_unit_price,
            min_ram=slots.ram_gb,
        )
        trace, relaxed = [], []

        for rung in range(max_rungs + 1):
            candidates = self.search(slots, constraints)
            passing, blocked = [], []

            for cand in candidates:
                scores = self.score(slots, cand)
                floor = self.violates_floor(slots, cand, scores)
                if floor:
                    blocked.extend(floor)
                elif scores["suitability_proba"] >= min_suitability:
                    passing.append({**cand, "scores": scores})

            trace.append({
                "rung": rung,
                "budget": constraints.effective_price(),
                "considered": len(candidates),
                "passing": len(passing),
            })

            if passing:
                passing.sort(key=lambda c: (-c["scores"]["suitability_proba"], c["price"]))
                return {
                    "status": "ok",
                    "options": passing[:3],
                    "relaxed": relaxed,
                    "trace": trace,
                }

            if rung == max_rungs:
                return {
                    "status": "infeasible",
                    "options": [],
                    "relaxed": relaxed,
                    "trace": trace,
                    "binding": self._binding_constraint(candidates, blocked, constraints),
                }

            key, description = RELAXATION_LADDER[rung]
            constraints = self._relax(constraints, key)
            relaxed.append(description)

        raise AssertionError("unreachable")

    @staticmethod
    def _relax(constraints, key):
        if key == "price_band":
            constraints.price_multiplier *= 1.15
        elif key == "drop_specs":
            constraints.relax_specs = True
        elif key == "refurbished":
            constraints.allow_refurbished = True
        elif key == "compute_tier":
            constraints.allow_lower_tier = True
        return constraints

    @staticmethod
    def _binding_constraint(candidates, blocked, constraints):
        """Name what actually stopped it -- the useful half of a refusal."""
        if not candidates:
            budget = constraints.effective_price()
            if budget is not None:
                return f"no catalogue configuration exists at or below {budget:,.0f} IDR"
            return "no catalogue configuration matched the requested specification"
        if blocked:
            top = max(set(blocked), key=blocked.count)
            return f"every affordable option {top}"
        return "no option cleared the suitability threshold for this role"


# --------------------------------------------------------------------------
# Rendering model output for the LLM
# --------------------------------------------------------------------------

# Raw probabilities are read inconsistently by an LLM; the bands are fixed once
# here so 0.56 does not become "definitely not" on one turn and "fine" on the next.
def suitability_band(p):
    if p >= 0.75:
        return "good fit"
    if p >= 0.55:
        return "acceptable, with caveats"
    if p >= 0.35:
        return "poor fit"
    return "not suitable"


def describe_scores(scores):
    lines = [
        f"- Suitability: {scores['suitability_proba']:.0%} ({suitability_band(scores['suitability_proba'])})",
        f"- Spec gap: {scores['spec_gap']:+.2f} "
        f"({'over-provisioned' if scores['spec_gap'] > 0.75 else 'under-provisioned' if scores['spec_gap'] < -0.75 else 'about right'})",
        f"- Expected lifespan: {scores['expected_lifespan_months']:.0f} months",
        f"- Expected 3-year operating cost: {scores['expected_opex_idr']:,.0f} IDR",
    ]
    if scores.get("approval_proba") is not None:
        lines.append(f"- Approval odds: {scores['approval_proba']:.0%} ({scores['approval_decision']})")
    else:
        blocked = ", ".join(scores.get("approval_blocked_on", []))
        lines.append(f"- Approval odds: not computed (needs {blocked})")
    return "\n".join(lines)


def describe_agent_result(result):
    lines = []
    if result["status"] == "ok":
        lines.append(f"Found {len(result['options'])} option(s).")
        for opt in result["options"]:
            row = opt["row"]
            lines.append(
                f"\n{row['laptop_brand']} {row['laptop_model']} — {opt['ram_gb']} GB, "
                f"{opt['price']:,.0f} IDR"
                + (" (refurbished)" if opt["refurbished"] else "")
            )
            lines.append(describe_scores(opt["scores"]))
    else:
        lines.append(f"No option satisfies the request. Binding constraint: {result['binding']}.")

    if result["relaxed"]:
        lines.append("\nConstraints relaxed to get here: " + "; ".join(result["relaxed"]) + ".")
    return "\n".join(lines)

## 4. Train the four models

CPU-only, so it runs before the GPU work. Hyperparameters are fixed at the values
`ml_applications.ipynb` found by `RandomizedSearchCV` — re-searching here would
add minutes on every cold start to rediscover them.

Which model won each task was decided on metrics, not by assuming XGBoost
everywhere: **ridge** wins UC2 because the generator's survival term is linear in
the features, so a linear model is correctly specified.

In [ ]:
import ml_models

bundle = ml_models.train_all(n_records=6000, out_dir=os.environ["MODEL_DIR"])

UC3 predicts **operating cost**, not total TCO. Total TCO is purchase price plus
operating cost, and purchase price is a *feature* — so a model predicting it
scores R² ≈ 0.99 by echoing an input back and tells finance nothing they did not
already know from the quote.

## 5. Build the Qdrant index

Runs **before** vLLM starts, so the embedding model has the whole GPU to itself.
Skip if the collection is already populated and `EMBED_MODEL` has not changed.

In [ ]:
import os
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import qdrant_client

load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_collection = os.getenv("QDRANT_COLLECTION", "bni_training")

# Must match app.py. Both read the same env var so the two cannot drift apart:
# a query embedded with a different model than the stored vectors still has the
# right dimensionality, so Qdrant accepts it and silently returns noise.
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL", "BAAI/bge-m3")  # 1024-dim, multilingual

# The collection holds vectors from one model only. Set RECREATE_COLLECTION=0 to
# append instead, but changing EMBED_MODEL without a wipe leaves the collection
# holding two incompatible vector spaces at once.
RECREATE_COLLECTION = os.getenv("RECREATE_COLLECTION", "1") not in ("0", "false", "False")

Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

client = qdrant_client.QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
    # The cluster is in sa-east-1; the 5s default is not enough for a
    # round trip with 1024-dim vectors and the writes time out.
    timeout=120,
    check_compatibility=False,
)
if RECREATE_COLLECTION and client.collection_exists(qdrant_collection):
    print(f"Deleting existing collection {qdrant_collection} so it is rebuilt with {EMBED_MODEL_NAME}.")
    client.delete_collection(qdrant_collection)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=qdrant_collection,
    batch_size=8,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


# --------------------------------------------------------------------------
# Source 1: PDF-derived JSON (pdfs/parsed_result and data/parsed_result)
# --------------------------------------------------------------------------

def _page_documents(json_path, pages):
    """parse.py output: a list of page objects with text/table blocks."""
    documents = []

    for page in pages:
        content = [block["text"] for block in page.get("text_blocks", [])]
        content.extend(
            "\n".join(" | ".join(row) for row in table["data"])
            for table in page.get("table_blocks", [])
        )

        if content:
            documents.append(Document(
                text="\n".join(content),
                metadata={
                    "source": json_path.name,
                    "page": page["page"],
                },
            ))

    return documents


def _flat_document(json_path, payload):
    """Flat {"text": "..."} output -- one blob, no page structure to keep."""
    text = payload.get("text", "").strip()
    if not text:
        return []

    return [Document(text=text, metadata={"source": json_path.name})]


def _purchase_request_document(json_path, payload):
    """parse_model.py output: a structured PurchaseRequest object.

    Rendered as readable "key: value" prose so the embedding model has real
    language to work with rather than raw JSON punctuation.
    """
    search = payload.get("search_summary") or {}

    lines = [
        f"Purchase request {payload.get('request_id', 'unknown')} "
        f"from {payload.get('department', 'unknown department')}.",
        f"Item category: {payload.get('item_category')}.",
        f"Specs searched: {search.get('specs_searched')}.",
        f"Quantity: {payload.get('quantity')} unit(s).",
        f"Requested unit price: {payload.get('requested_unit_price')} IDR.",
        f"Historical average price: {payload.get('historical_avg_price')} IDR.",
        f"Price variance ratio: {payload.get('price_variance_ratio')}.",
        f"Total amount: {payload.get('total_amount')} IDR.",
        f"Cheapest vendor found: {search.get('cheapest_vendor_found')} "
        f"via {search.get('vendor_channel_type')}.",
        f"Real price found: {search.get('real_price')} IDR "
        f"(saving {search.get('price_savings_vs_requested')} IDR vs requested).",
        f"Vendor risk score: {payload.get('vendor_risk_score')}.",
        f"Department budget remaining: {payload.get('dept_budget_remaining')} IDR.",
        f"Urgent: {'yes' if payload.get('is_urgent') else 'no'}.",
    ]

    return [Document(
        text="\n".join(lines),
        metadata={
            "source": json_path.name,
            "doc_type": "purchase_request",
            "request_id": payload.get("request_id"),
            "department": payload.get("department"),
        },
    )]


def load_parsed_documents(json_paths):
    documents = []

    for json_path in json_paths:
        try:
            payload = json.loads(json_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f"  {json_path.name}: skipped, unreadable ({e})")
            continue

        if isinstance(payload, list):
            found = _page_documents(json_path, payload)
        elif isinstance(payload, dict) and "text" in payload:
            found = _flat_document(json_path, payload)
        elif isinstance(payload, dict) and "request_id" in payload:
            found = _purchase_request_document(json_path, payload)
        else:
            print(f"  {json_path.name}: skipped, unrecognised shape ({type(payload).__name__})")
            continue

        print(f"  {json_path.name}: {len(found)} document(s)")
        documents.extend(found)

    return documents


# --------------------------------------------------------------------------
# Source 2: laptop listings (data/laptop_listings_filled.json)
# --------------------------------------------------------------------------

def _listing_text(listing):
    """Render one marketplace listing as prose for embedding."""
    spec_bits = []
    for label, key, unit in [
        ("CPU", "cpu", ""),
        ("RAM", "ram_gb", " GB"),
        ("Storage", "storage_gb", " GB"),
        ("GPU", "gpu", ""),
        ("Screen", "screen_size_in", " inch"),
    ]:
        value = listing.get(key)
        if value is not None:
            spec_bits.append(f"{label}: {value}{unit}")

    lines = [
        listing.get("title") or "Untitled listing",
        f"Brand: {listing.get('brand')}. Model: {listing.get('model')}.",
    ]
    if spec_bits:
        lines.append("Specifications -- " + ", ".join(spec_bits) + ".")

    lines.append(
        f"Price: {listing.get('price_idr')} IDR"
        + (f" (original {listing['original_price_idr']} IDR)"
           if listing.get("original_price_idr") is not None else "")
        + f". Condition: {listing.get('condition')}."
    )
    lines.append(
        f"Sold by {listing.get('seller_name')} on {listing.get('source')} "
        f"({'official store' if listing.get('seller_is_official') else 'regular seller'}), "
        f"located in {listing.get('location')}."
    )

    if listing.get("seller_rating") is not None:
        lines.append(
            f"Seller rating: {listing['seller_rating']}"
            + (f" from {listing['seller_num_reviews']} reviews"
               if listing.get("seller_num_reviews") is not None else "")
            + "."
        )

    if listing.get("is_suspected_scam"):
        reasons = ", ".join(listing.get("scam_reasons") or []) or "unspecified"
        lines.append(f"WARNING: flagged as a suspected scam listing. Reasons: {reasons}.")
    else:
        lines.append("Not flagged as a suspected scam listing.")

    return "\n".join(lines)


def load_listing_documents(listings_path):
    listings = json.loads(Path(listings_path).read_text(encoding="utf-8"))

    documents = [
        Document(
            text=_listing_text(listing),
            metadata={
                "source": Path(listings_path).name,
                "doc_type": "laptop_listing",
                "marketplace": listing.get("source"),
                "source_id": listing.get("source_id"),
                "brand": listing.get("brand"),
                "price_idr": listing.get("price_idr"),
                "is_suspected_scam": bool(listing.get("is_suspected_scam")),
                "url": listing.get("url"),
            },
        )
        for listing in listings
    ]

    print(f"  {Path(listings_path).name}: {len(documents)} listing document(s)")
    return documents


# --------------------------------------------------------------------------
# Source 3: BNI training notes (data/BNI/*.txt) -- hierarchical indexing
# --------------------------------------------------------------------------

# Split on a sentence terminator followed by whitespace and a capital/quote.
_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9])")

# Children shorter than this are dropped as headings/labels rather than embedded.
MIN_CHILD_CHARS = 20


def _split_sentences(paragraph):
    """Split a paragraph into child units.

    The notes are line-oriented -- headings and bullet lines often carry no
    terminal punctuation -- so split on newlines first, then on sentence
    boundaries within each line.
    """
    sentences = []
    for line in paragraph.splitlines():
        line = line.strip()
        if line:
            sentences.extend(s.strip() for s in _SENTENCE_RE.split(line) if s.strip())
    return sentences


def load_bni_hierarchical_nodes(txt_paths):
    """Build child (sentence) nodes that each carry their parent paragraph.

    Only the sentence text is embedded, so retrieval matches at sentence
    granularity. The full parent paragraph rides along in metadata as
    `parent_text`, which the query side feeds to the LLM as context.
    """
    nodes = []

    for txt_path in txt_paths:
        raw = txt_path.read_text(encoding="utf-8", errors="replace").strip()
        if not raw:
            print(f"  {txt_path.name}: skipped, file is empty")
            continue

        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", raw) if p.strip()]
        file_sentences = 0

        for para_index, paragraph in enumerate(paragraphs):
            parent_id = f"{txt_path.stem}-p{para_index}"
            sentences = _split_sentences(paragraph)

            # Bare headings and labels ("Day 1", "Treasury") are too short to
            # match on usefully and only add retrieval noise; their paragraph
            # stays reachable through its longer sentences. If a paragraph is
            # nothing but short lines, keep it whole so it is still findable.
            long_enough = [s for s in sentences if len(s) >= MIN_CHILD_CHARS]
            sentences = long_enough or [paragraph]

            for sent_index, sentence in enumerate(sentences):
                node = TextNode(
                    text=sentence,
                    metadata={
                        "source": txt_path.name,
                        "doc_type": "bni_training",
                        "week": txt_path.stem,
                        "parent_id": parent_id,
                        "parent_index": para_index,
                        "sentence_index": sent_index,
                        "parent_text": paragraph,
                    },
                )
                # Embed the sentence alone -- the parent paragraph is context
                # for the LLM, not part of what the retriever matches against.
                node.excluded_embed_metadata_keys = list(node.metadata.keys())
                nodes.append(node)

            file_sentences += len(sentences)

        print(f"  {txt_path.name}: {len(paragraphs)} paragraph(s) -> {file_sentences} sentence node(s)")

    return nodes


# --------------------------------------------------------------------------
# Build the index
# --------------------------------------------------------------------------

# Hardcoding each source's path broke every time the data moved, and on Kaggle
# everything is flattened into one dataset dir. Instead, search a few plausible
# roots recursively and classify each file by name.
_kaggle_input = Path("/kaggle/input")
ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/revelelel/parsed-result"),
    Path("/kaggle/input/parsed-result"),
    *(sorted(_kaggle_input.glob("*")) if _kaggle_input.is_dir() else []),
    Path.cwd() / "data",
    Path.cwd(),
]

roots = []
for root in ROOT_CANDIDATES:
    if root.is_dir() and not any(root == seen or root in seen.parents for seen in roots):
        roots.append(root)

if not roots:
    raise FileNotFoundError("No data root found. Checked: "
                            + ", ".join(str(p) for p in ROOT_CANDIDATES))


def discover_sources(roots):
    """Classify every file under the roots into one of the three sources."""
    bni_files, listing_files, parsed_files = [], [], []
    seen_names = set()

    for root in roots:
        for path in sorted(root.rglob("*")):
            if not path.is_file() or path.stat().st_size == 0:
                continue

            # The same filename showing up under two roots is the same data
            # mounted twice, not two documents.
            if path.name in seen_names:
                continue

            if path.suffix.lower() == ".txt":
                bni_files.append(path)
            elif path.suffix.lower() == ".json":
                if path.stem.lower().startswith("laptop_listings"):
                    listing_files.append(path)
                else:
                    parsed_files.append(path)
            else:
                continue

            seen_names.add(path.name)

    # laptop_listings_filled.json supersedes the un-enriched laptop_listings.json.
    if any(p.stem.lower().endswith("_filled") for p in listing_files):
        listing_files = [p for p in listing_files if p.stem.lower().endswith("_filled")]

    return bni_files, listing_files, parsed_files


print("Searching for data under:", ", ".join(str(r) for r in roots))
bni_files, listing_files, parsed_files = discover_sources(roots)

all_nodes = []
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# 1. PDF-derived parsed results.
if parsed_files:
    print(f"\nParsed PDF results ({len(parsed_files)} file(s)):")
    parsed_docs = load_parsed_documents(parsed_files)
    parsed_nodes = splitter.get_nodes_from_documents(parsed_docs)
    all_nodes.extend(parsed_nodes)
    print(f"  -> {len(parsed_nodes)} chunk(s)")
else:
    print("\nNo parsed PDF JSON found.")

# 2. Laptop listings.
if listing_files:
    print(f"\nLaptop listings ({len(listing_files)} file(s)):")
    listing_docs = []
    for path in listing_files:
        listing_docs.extend(load_listing_documents(path))
    listing_nodes = splitter.get_nodes_from_documents(listing_docs)
    all_nodes.extend(listing_nodes)
    print(f"  -> {len(listing_nodes)} chunk(s)")
else:
    print("\nNo laptop listings JSON found.")

# 3. BNI training notes, hierarchically indexed (sentence child / paragraph parent).
if bni_files:
    print(f"\nBNI training notes ({len(bni_files)} file(s)):")
    bni_nodes = load_bni_hierarchical_nodes(bni_files)
    all_nodes.extend(bni_nodes)
    print(f"  -> {len(bni_nodes)} sentence chunk(s)")
else:
    print("\nNo BNI .txt notes found.")

if not all_nodes:
    raise RuntimeError("No documents found in any configured source.")

print(f"\nCreated {len(all_nodes)} embedding chunks across all sources.")

VectorStoreIndex(all_nodes, storage_context=storage_context)

print("Qdrant collections:", [item.name for item in client.get_collections().collections])
print(f"Done. Embeddings are stored in the {qdrant_collection} collection.")

## 6. Release the GPU

`HuggingFaceEmbedding` loaded bge-m3 into *this kernel's* VRAM and it stays
resident until dropped. vLLM sizes its KV cache against free memory at startup,
so anything still held here is memory the server never gets.

In [ ]:
import gc

import torch
from llama_index.core import Settings

# The private attribute deliberately: the public `Settings.embed_model` setter
# runs its argument through resolve_embed_model(), and resolve_embed_model(None)
# returns the *default* model rather than clearing the field -- so assigning None
# through the setter would leave a model loaded instead of freeing one.
Settings._embed_model = None
gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

## 7. Test the router and agent — before any LLM is involved

Everything except extraction is deterministic Python, so it can be exercised
without the model server running. If these pass and the app still misbehaves, the
fault is in extraction or generation, not in the routing.

In [ ]:
from ml_models import load_all, score_request
from procurement_agent import (
    GATE_DESCRIPTIONS, RequestSlots, SourcingAgent, catalogue_row,
    describe_agent_result, normalise_cpu, normalise_role, route, validate_slots,
)

bundle = load_all(os.environ["MODEL_DIR"])

print("=== G0: the 'Macbook intel i7' case ===")
slots = RequestSlots(department="Design", seniority_level="Manager",
                     laptop_brand="Apple", laptop_model="MacBook Pro 14",
                     ram_gb=32, intent="assess")
slots._cpu_text = "intel i7"
slots.cpu = normalise_cpu("intel i7", "Apple")

validation = validate_slots(slots)
print("cpu resolved to :", slots.cpu)
print("gate            :", route(slots)[0])
for correction in validation.corrections:
    print("  ->", correction)

print("\n=== Indonesian role abbreviations ===")
for text in ["buat AMGR Design", "untuk Officer Design", "buat direktur"]:
    print(f"  {text!r:26} -> {normalise_role(text)}")

In [ ]:
print("=== router gates ===")
cases = [
    ("named machine, complete", RequestSlots(
        department="Design", seniority_level="Manager", laptop_brand="Apple",
        laptop_model="MacBook Pro 14", cpu="M3 Pro", ram_gb=32,
        requested_unit_price=36_000_000, intent="assess")),
    ("goal + budget", RequestSlots(
        department="Design", seniority_level="Staff",
        dept_budget_remaining=5_000_000, intent="find")),
    ("document question", RequestSlots(intent="ask")),
    ("whole month", RequestSlots(intent="batch")),
    ("budget, no department", RequestSlots(intent="find", dept_budget_remaining=5_000_000)),
    ("nothing yet", RequestSlots()),
]
for label, case in cases:
    gate, _ = route(case)
    print(f"  {label:24} -> {gate:8} {GATE_DESCRIPTIONS[gate]}")

### The sourcing agent's loop

Three runs: one that succeeds immediately, one that walks the whole ladder and
still fails, and one where the hard floor — not the budget — is what blocks it.
The refusal is the interesting case: an agent that returns "nothing found" is
useless, while one that names the binding constraint is actionable.

In [ ]:
agent = SourcingAgent(bundle)

scenarios = [
    ("Design Manager, 35 juta", RequestSlots(
        department="Design", seniority_level="Manager",
        dept_budget_remaining=35_000_000, intent="find")),
    ("Sales Staff, 15 juta", RequestSlots(
        department="Sales", seniority_level="Staff",
        dept_budget_remaining=15_000_000, intent="find")),
    ("Design Staff, 9 juta", RequestSlots(
        department="Design", seniority_level="Staff",
        dept_budget_remaining=9_000_000, intent="find")),
]

for label, slots in scenarios:
    result = agent.run(slots)
    print(f"\n{'=' * 68}\n{label}  ->  {result['status']}\n{'=' * 68}")
    print("rungs walked:",
          [(t["rung"], f"{t['considered']} considered", f"{t['passing']} passing")
           for t in result["trace"]])
    print(describe_agent_result(result)[:900])

## 8. Start the local LLM server

vLLM runs as a **separate process** — not in this kernel, and not inside
`app.py`. Streamlit re-executes its whole script on every widget interaction, so
a model loaded there would reload per session and fight the embedding model for
VRAM. Here it loads once and answers over HTTP.

First run downloads ~10 GB.

In [ ]:
import subprocess
import time

import requests

VLLM_PORT = 8000
MODEL = os.environ["HF_MODEL"]


def launch_vllm():
    # Re-running this cell leaves the old server holding the port and the VRAM.
    subprocess.run(["pkill", "-f", "vllm.entrypoints"], check=False)
    time.sleep(5)

    log = open("vllm.log", "w")
    proc = subprocess.Popen(
        [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", MODEL,
            # The app sends HF_MODEL as the model name and vLLM rejects a
            # mismatch, so the served name must be this exact string.
            "--served-model-name", MODEL,
            "--port", str(VLLM_PORT),
            # T4s are compute capability 7.5 and have no bfloat16.
            "--dtype", "half",
            "--tensor-parallel-size", "2",
            "--max-model-len", "8192",
            "--gpu-memory-utilization", "0.90",
        ],
        stdout=log, stderr=subprocess.STDOUT,
    )

    for _ in range(1200):
        if proc.poll() is not None:
            raise RuntimeError("vLLM exited early:\n" + open("vllm.log").read()[-4000:])
        try:
            if requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2).ok:
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    else:
        raise RuntimeError("vLLM never came up:\n" + open("vllm.log").read()[-4000:])

    print(f"vLLM is serving {MODEL} on http://localhost:{VLLM_PORT}/v1")
    return proc


vllm_proc = launch_vllm()

## 9. Test extraction — the weakest link in the chain

Everything downstream assumes the extracted request is faithful. It is the one
step with no ground truth to check against, and its failures are quiet: a wrong
department produces a confident, precise, entirely wrong set of numbers.

Two defences, both visible below. **Guided decoding** pins the output to a JSON
schema whose enums are the catalogue's own vocabulary, so the model cannot invent
a department. **G0** then rejects combinations that are individually valid but
jointly impossible.

In [ ]:
from openai import OpenAI

from procurement_agent import extract_slots

llm = OpenAI(base_url=os.environ["LOCAL_LLM_URL"], api_key="EMPTY", timeout=600)
print("Served models:", [m.id for m in llm.models.list().data])

messages = [
    "Beli Macbook intel i7 dan ram 32gb buat AMGR Design",
    "Cari laptop buat Officer Design, budget 5 juta",
    "Biasanya divisi Design beli laptop apa?",
    "Review semua request bulan ini",
]

for message in messages:
    slots = extract_slots(llm, MODEL, message, RequestSlots())
    gate, validation = route(slots)
    known = {k: v for k, v in vars(slots).items()
             if v is not None and not k.startswith("_") and k != "free_text"}
    print(f"\n{message}")
    print(f"  gate  : {gate}")
    print(f"  slots : {known}")
    for correction in validation.corrections:
        print(f"  fix   : {correction}")

## 10. One turn end to end

Extraction → routing → the lane's own work → narration. This is exactly what
`app.py` does per turn; running it here makes the failure legible when something
breaks, instead of surfacing as a red box in Streamlit.

In [ ]:
from procurement_agent import describe_scores

SYSTEM = (
    "You are a procurement assistant for BNI. Numbers in a MODEL OUTPUT block "
    "come from trained models and are authoritative -- explain them, never "
    "recompute or contradict them. Be concise. Reply in the language of the question."
)


def answer_one_turn(message, slots=None, temperature=0.3):
    slots = extract_slots(llm, MODEL, message, slots or RequestSlots())
    gate, validation = route(slots)

    if gate == "G0":
        parts = list(validation.corrections)
        if validation.missing:
            parts.append("Still needed: " + ", ".join(validation.missing) + ".")
        turn = "TASK: relay these catalogue corrections and ask for what is missing.\n\n" + "\n".join(parts)
    elif gate == "G2":
        turn = (f"MODEL OUTPUT:\n{describe_scores(score_request(bundle, slots.as_scoring_fields()))}"
                f"\n\nExplain this. User said: {message}")
    elif gate == "G3":
        turn = (f"MODEL OUTPUT from the sourcing agent:\n{describe_agent_result(agent.run(slots))}"
                f"\n\nPresent the options and say what was relaxed. User said: {message}")
    else:
        turn = f"TASK: answer briefly. User said: {message}"

    reply = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": turn}],
        max_tokens=600, temperature=temperature,
    )
    return gate, slots, reply.choices[0].message.content


for message in ["Beli Macbook intel i7 dan ram 32gb buat AMGR Design",
                "Cari laptop buat Manager Design, budget 35 juta"]:
    gate, slots, reply = answer_one_turn(message)
    print(f"\n{'=' * 70}\nUSER  {message}\nGATE  {gate}\n{'=' * 70}\n{reply}")

## 11. Write `app.py`

The same modules the cells above exercised, wrapped in a chat UI. It carries no
credentials and no routing rules of its own — it imports `procurement_agent` and
reads the environment set in step 2, which the Streamlit subprocess inherits.

The sidebar shows the **request under construction**, which is the thing that
makes multi-turn slot filling legible: the laptop arrives in one turn and the
budget in another, and you can watch the object fill up.

In [ ]:
%%writefile app.py
"""Streamlit UI for the procurement assistant.

One chat box, three lanes behind it. Each turn is extracted into a structured
request, validated against the catalogue, and routed:

    G0  catalogue mismatch / missing field  -> correct and ask, no model call
    G1  question about the documents        -> RAG over Qdrant
    G2  one resolved candidate              -> ML scoring, narrated
    G3  goal plus constraint, no candidate  -> sourcing agent (plan/search/replan)
    G4  many requests at once               -> batch lane
    --  nothing matched                     -> clarify

The routing and agent logic live in procurement_agent.py so this file and the
notebook cannot drift apart -- the two used to declare their own config and
ended up pointing at different Qdrant clusters.

Run with: streamlit run app.py
On Kaggle, start it from a notebook cell and open the tunnel there.
"""

import os

import qdrant_client
import streamlit as st
from dotenv import load_dotenv
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore

from ml_models import load_all, score_request
from procurement_agent import (
    GATE_DESCRIPTIONS, RequestSlots, SourcingAgent,
    describe_agent_result, describe_scores, extract_slots, route,
)

load_dotenv()

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
QDRANT_COLLECTION = os.environ.get("QDRANT_COLLECTION", "capstone")
HF_MODEL = os.environ.get("HF_MODEL", "Qwen/Qwen2.5-14B-Instruct-AWQ")
HF_TOKEN = os.environ.get("HF_TOKEN")
LLM_BACKEND = os.environ.get("LLM_BACKEND", "local").lower()
LOCAL_LLM_URL = os.environ.get("LOCAL_LLM_URL", "http://localhost:8000/v1")
MODEL_DIR = os.environ.get("MODEL_DIR", "models")

EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL", "BAAI/bge-m3")
# bge-m3 embeds one short query per turn, which the CPU handles in well under a
# second. Keeping it off the GPU leaves the whole card to vLLM, which sizes its
# KV cache once at startup and cannot give memory back.
EMBED_DEVICE = os.environ.get("EMBED_DEVICE", "cpu")

SYSTEM_PROMPT = (
    "You are a procurement assistant for BNI. You answer about training notes, "
    "laptop procurement documents, and marketplace listings.\n"
    "When the message includes a MODEL OUTPUT block, those numbers come from "
    "trained models and are authoritative -- explain them, never recompute or "
    "contradict them. When it includes a CONTEXT block, answer only from it and "
    "say you don't know if the answer isn't there.\n"
    "Be concise. Reply in the same language as the question."
)

st.set_page_config(page_title="Procurement Assistant", page_icon="🔀", layout="centered")


# --------------------------------------------------------------------------
# Resources
# --------------------------------------------------------------------------

@st.cache_resource(show_spinner="Loading embedding model and connecting to Qdrant...")
def get_index():
    Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME, device=EMBED_DEVICE)
    client = qdrant_client.QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
    store = QdrantVectorStore(client=client, collection_name=QDRANT_COLLECTION)
    return VectorStoreIndex.from_vector_store(store)


@st.cache_resource(show_spinner="Loading procurement models...")
def get_bundle():
    return load_all(MODEL_DIR)


@st.cache_resource
def get_llm_client():
    if LLM_BACKEND == "local":
        from openai import OpenAI

        # vLLM ignores the key but the client refuses to construct without one.
        return OpenAI(base_url=LOCAL_LLM_URL, api_key="EMPTY", timeout=600)

    from openai import OpenAI

    return OpenAI(api_key=HF_TOKEN, base_url="https://api-inference.huggingface.co/v1")


def stream_chat(client, messages, temperature):
    stream = client.chat.completions.create(
        model=HF_MODEL, messages=messages, max_tokens=900,
        temperature=temperature, stream=True,
    )
    for chunk in stream:
        choices = getattr(chunk, "choices", None) or []
        if choices:
            yield getattr(choices[0].delta, "content", None) or ""


# --------------------------------------------------------------------------
# RAG helpers
# --------------------------------------------------------------------------

def expand_to_parents(nodes):
    """Trade each retrieved sentence for the paragraph it came from.

    The BNI notes are indexed hierarchically: a sentence is embedded as the
    child with its whole paragraph carried in `parent_text`. Matching happens on
    the sentence; the LLM should read the paragraph. Sentences from one
    paragraph collapse so the same text is not sent twice.
    """
    chunks, seen = [], set()
    for scored in nodes:
        node = scored.node
        parent = node.metadata.get("parent_text")
        if parent is None:
            chunks.append(node.get_content())
            continue
        key = node.metadata.get("parent_id", parent)
        if key not in seen:
            seen.add(key)
            chunks.append(parent)
    return chunks


def format_source_label(meta):
    source = meta.get("source", "?")
    kind = meta.get("doc_type")
    if kind == "bni_training":
        return f"{source} — paragraph {meta.get('parent_index', '?')}"
    if kind == "laptop_listing":
        return f"{source} — {meta.get('brand', '?')} listing {meta.get('source_id', '?')}"
    if kind == "purchase_request":
        return f"{source} — request {meta.get('request_id', '?')}"
    if meta.get("page") is not None:
        return f"{source} — page {meta['page']}"
    return source


def retrieve(index, question, top_k):
    nodes = index.as_retriever(similarity_top_k=top_k).retrieve(question)
    return nodes, expand_to_parents(nodes)


# --------------------------------------------------------------------------
# Lanes
# --------------------------------------------------------------------------

def lane_g0(validation):
    """No model call: the catalogue already knows this configuration is wrong."""
    parts = list(validation.corrections)
    if validation.missing:
        parts.append("Still needed: " + ", ".join(validation.missing) + ".")
    return "TASK: relay these catalogue corrections and ask for what is missing.\n\n" + "\n".join(parts)


def lane_g1(index, question, top_k):
    nodes, chunks = retrieve(index, question, top_k)
    context = "\n\n---\n\n".join(chunks) if chunks else "(no matching context found)"
    return f"CONTEXT:\n{context}\n\nQuestion: {question}", nodes


def lane_g2(bundle, slots):
    scores = score_request(bundle, slots.as_scoring_fields())
    block = describe_scores(scores)
    ask = ""
    if scores.get("approval_proba") is None:
        ask = ("\nThe approval model needs "
               + ", ".join(scores.get("approval_blocked_on", []))
               + " -- ask the user for it before promising approval odds.")
    return (f"MODEL OUTPUT for the requested machine:\n{block}\n{ask}\n\n"
            f"Explain this to the user. If the spec gap is positive and large, say the "
            f"configuration is over-specified for the role.\n\nUser said: {slots.free_text}"), scores


def lane_g3(bundle, slots, index, top_k):
    result = SourcingAgent(bundle).run(slots)
    block = describe_agent_result(result)
    nodes, chunks = ([], [])
    if index is not None:
        try:
            nodes, chunks = retrieve(index, slots.free_text, top_k)
        except Exception:
            pass
    context = ("\n\nRELATED PAST PURCHASES / LISTINGS:\n" + "\n\n".join(chunks[:3])) if chunks else ""
    return (f"MODEL OUTPUT from the sourcing agent:\n{block}{context}\n\n"
            f"Present the options (or explain why none exists). State plainly which "
            f"constraints were relaxed.\n\nUser said: {slots.free_text}"), result, nodes


# --------------------------------------------------------------------------
# UI
# --------------------------------------------------------------------------

index = get_index()
bundle = get_bundle()
llm = get_llm_client()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "slots" not in st.session_state:
    # Slots accumulate across turns: the laptop arrives in one turn and the
    # budget in another. One home for the object, or the router reads a
    # half-built request and mis-gates it.
    st.session_state.slots = RequestSlots()

with st.sidebar:
    st.header("Settings")
    top_k = st.slider("Retrieved chunks", 1, 10, 5)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.3, 0.05)
    show_trace = st.checkbox("Show routing trace", value=True)
    if st.button("Clear chat and request"):
        st.session_state.messages = []
        st.session_state.slots = RequestSlots()
        st.rerun()

    st.divider()
    st.caption("Request under construction")
    known = {k: v for k, v in vars(st.session_state.slots).items()
             if v is not None and not k.startswith("_") and k != "free_text"}
    st.json(known or {"(empty)": None}, expanded=True)

st.title("🔀 Procurement Assistant")
st.caption(f"`{QDRANT_COLLECTION}` · {HF_MODEL} · router + 4 ML models")

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

question = st.chat_input("Contoh: beli Macbook ram 32gb buat AMGR Design")

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        placeholder = st.empty()
        nodes, extra = [], None

        try:
            with st.spinner("Reading the request..."):
                slots = extract_slots(llm, HF_MODEL, question, st.session_state.slots)
                st.session_state.slots = slots
                gate, validation = route(slots)

            if gate == "G0":
                user_turn = lane_g0(validation)
            elif gate == "G1":
                user_turn, nodes = lane_g1(index, question, top_k)
            elif gate == "G2":
                user_turn, extra = lane_g2(bundle, slots)
            elif gate == "G3":
                with st.spinner("Sourcing agent working..."):
                    user_turn, extra, nodes = lane_g3(bundle, slots, index, top_k)
            elif gate == "G4":
                user_turn = ("TASK: tell the user batch review is not wired up yet, "
                             "and offer to assess one request at a time.")
            else:
                user_turn = (f"TASK: ask what the user needs. Known so far: "
                             f"{slots.as_scoring_fields() or 'nothing'}.")

            chat = [{"role": "system", "content": SYSTEM_PROMPT}]
            chat += [{"role": m["role"], "content": m["content"]}
                     for m in st.session_state.messages[-7:-1]]
            chat.append({"role": "user", "content": user_turn})

            answer = ""
            for delta in stream_chat(llm, chat, temperature):
                answer += delta
                if answer:
                    placeholder.markdown(answer + "▌")
            placeholder.markdown(answer or "_(no response)_")

        except Exception as e:
            gate = locals().get("gate", "?")
            answer = f"Error in lane {gate}: {type(e).__name__}: {e}"
            placeholder.markdown(answer)

        if show_trace:
            with st.expander(f"Routing trace — {gate}", expanded=False):
                st.caption(GATE_DESCRIPTIONS.get(gate, ""))
                if extra and isinstance(extra, dict) and "trace" in extra:
                    st.caption("Sourcing agent rungs")
                    st.dataframe(extra["trace"], use_container_width=True)
                    if extra.get("relaxed"):
                        st.caption("Relaxed: " + "; ".join(extra["relaxed"]))
                    if extra.get("binding"):
                        st.warning("Binding constraint: " + extra["binding"])
                elif extra:
                    st.json({k: v for k, v in extra.items() if k != "assumed_fields"})
                    if extra.get("assumed_fields"):
                        st.caption("Assumed (not stated by the user): "
                                   + ", ".join(extra["assumed_fields"]))

        if nodes:
            with st.expander("Sources"):
                for n in nodes:
                    meta = n.node.metadata
                    st.markdown(f"**{format_source_label(meta)}** (score: {n.score:.3f})")
                    parent = meta.get("parent_text")
                    st.text((parent or n.node.get_content())[:900])

    st.session_state.messages.append({"role": "assistant", "content": answer})

## 12. Launch the UI

The tunnel is opened here rather than inside `app.py` for the same reason the
model is: this cell runs once, whereas `app.py` re-runs on every interaction and
would open a duplicate tunnel each time.

Interrupt the cell to stop the app.

In [ ]:
from pyngrok import ngrok

STREAMLIT_PORT = 8501


def launch_streamlit():
    # Re-running the cell leaves the old server holding the port, which is what
    # "Port 8501 is not available" means. Clear both sides before starting.
    ngrok.kill()
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    time.sleep(2)

    log = open("streamlit.log", "w")
    proc = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            f"--server.port={STREAMLIT_PORT}",
            "--server.headless=true",
            "--server.address=0.0.0.0",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
            "--browser.gatherUsageStats=false",
        ],
        stdout=log, stderr=subprocess.STDOUT,
    )

    for _ in range(240):
        if proc.poll() is not None:
            raise RuntimeError("Streamlit exited early:\n" + open("streamlit.log").read())
        try:
            requests.get(f"http://localhost:{STREAMLIT_PORT}", timeout=2)
            break
        except requests.exceptions.RequestException:
            time.sleep(1)
    else:
        raise RuntimeError("Streamlit never came up:\n" + open("streamlit.log").read())

    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
    url = ngrok.connect(addr=STREAMLIT_PORT, proto="http").public_url
    url = url.replace("http://", "https://", 1)
    print("Streamlit is up.")
    print("Public URL:", url)
    return proc, url


streamlit_proc, public_url = launch_streamlit()
# Keep the cell alive -- the tunnel dies when this process exits.
streamlit_proc.wait()

## 13. Shut down

Frees both ports and the GPU without restarting the session.

In [ ]:
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["pkill", "-f", "vllm.entrypoints"], check=False)
ngrok.kill()
print("Stopped.")

---

### What is and is not built

**Working:** the G0 catalogue gate, the ordered router, all four models, the
sourcing agent's loop with its relaxation ladder and hard floor, multi-turn slot
accumulation, and schema-guided extraction.

**Not built:** the G4 batch lane is a stub. It reuses scoring and evaluation but
never relaxes anything, so it is probably a separate agent borrowing the same
tools rather than this one with a flag — worth deciding before writing it.

**The honest caveat:** the training data is synthetic. Every metric here
demonstrates a working pipeline, not a finding about real procurement. Swap in
historical data and the same modules re-fit unchanged.